## 04 Tools
Tools provide the "action" part of a ReACT agent. Their results are called _observations_. You can define tools yourselves or use an existing library of tools.

Following is an example of a tool, which multiplied 2 numbers and returns the product:

```python
from langchain.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """multiply 2 numbers

    Args:
      a (int): the first number to multiply
      b (int): the second number to multiply
    Result:
      int: product of the input numbers
    """
    return a * b 
```

The **LLM will use the description of the tool to decide when to use that tool, and the parameters of the tool and the response from tool**. The function itself will be executed by the `ToolNode`. Tools allow real-world agents to ACT. Careful description help your agent on how to use the tools and how to call them effectively. 

LangChain supports many tools formats and tool sets. In the example below, we define a tool that performs basic arithmetic on it's floating point parameters.

In [19]:
from dotenv import load_dotenv
from rich.console import Console
from rich.markdown import Markdown
from typing import Literal, Union

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool

load_dotenv(override=True)
console = Console()

In [20]:
@tool
def real_number_calculator(
    a: float,
    b: float,
    operation: Literal["add", "subtract", "multiply", "divide"],
) -> float | str:
    """Perform basic arithmatic operations on two real numbers."""
    # Args:
    #     a (float): The first number.
    #     b (float): The second number.
    #     operation (str): The operation to perform. One of 'add', 'subtract', 'multiply', 'divide'.
    # Returns:
    #     (float or str): The result of the operation.
    #     NOTE: ONLY a divide by zero will return a string "Cannot divide by zero!". All other valid
    #     operations will return a float or int.
    # """

    # NOTE: docstring is very basic by design!
    print(f"Invoking real_number_calculator to perform '{operation}' on {a} and {b}")

    if operation == "add":
        print(f"Adding {a} and {b}")
        return a + b
    elif operation == "subtract":
        print(f"Subtracting {b} from {a}")
        return a - b
    elif operation == "multiply":
        print(f"Multiplying {a} and {b}")
        return a * b
    elif operation == "divide":
        print(f"Dividing {a} by {b}")
        if b == 0:
            return "Cannot divide by zero!"
        return a / b
    else:
        raise ValueError(f"Unsupported operation: {operation}")

In [21]:
# create our agent using OpenAI LLM

openai_llm = init_chat_model(
    "gpt-4o-mini",  # replace this with any supported OpenAI model name
    model_provider="openai",
    temperature=0.7,
)

# agent = create_agent(
#     model=openai_llm,
#     system_prompt="""
#         You are a simple calculator that can perform simple math operations on real numbers.
#         You have access to a single tool, the `real_number_calculator`, which can perform addition, subtraction, multiplication, and division on two real numbers.
#         ALWAYS use this tool to perform the math operation EVEN IF you can do it without the tool.
#         Return a crisp response with just the final answer and no additional text.
#     """,
#     tools=[real_number_calculator],
# )

agent = create_agent(
    model=openai_llm,
    system_prompt="""
        You are a helpful assistant.
    """,
    tools=[real_number_calculator],
)

In [22]:
# helper function
def ask_agent(agent, query: str) -> str:
    response = agent.invoke({"messages": {"role": "user", "content": query}})
    return response["messages"][-1].content

In [23]:
while True:
    query = input("Enter a math query (or 'exit' to quit): ")
    if query.lower().strip() == "exit":
        break
    result = ask_agent(agent, query)
    print(result)

Invoking real_number_calculator to perform 'multiply' on 4.0 and 3.245
Multiplying 4.0 and 3.245
The result of \( 4.0 \times 3.245 \) is \( 12.98 \).


**NOTE**: The tool description can have a big impact. It may not invoke your calculator tool. Most modern LLMs, such as GPT, Claude and Gemini, are perfectly capable of performing these math operations without the help of tools. 

While basic descriptions suffice for most situations, LangChain has support for enhanced descriptions

In [29]:
@tool(
    "calculator",
    parse_docstring=True,
    description=(
        "Perform basic math operations on two real numbers."
        "Use this whenever you have operations on two numbers, even if they are integers."
    ),
)
def real_number_calculator2(
    a: float,
    b: float,
    operation: Literal["add", "subtract", "multiply", "divide"],
) -> float:
    """Perform basic math operations on two real numbers.

    Args:
        a (float): The first number.
        b (float): The second number.
        operation (Literal["add", "subtract", "multiply", "divide"]):
            The math operation to perform.
            - "add" - returns the sum of a and b
            - "subtract" - returns the result of subtracting b from a
            - "multiply" - returns the product of a and b
            - "divide" - returns the quotient of a and b

    Returns:
        (float): The numerical result of the operation.

    Raises:
        ValueError: If a division by 0 is attempted or if an unsupported operation is provided.

    """
    print(f"Invoking real_number_calculator2 to perform '{operation}' on {a} and {b}")

    if operation == "add":
        print(f"Adding {a} and {b}")
        return a + b
    elif operation == "subtract":
        print(f"Subtracting {b} from {a}")
        return a - b
    elif operation == "multiply":
        print(f"Multiplying {a} and {b}")
        return a * b
    elif operation == "divide":
        print(f"Dividing {a} by {b}")
        if b == 0:
            return "Cannot divide by zero!"
        return a / b
    else:
        raise ValueError(f"Unsupported operation: {operation}")()

In [30]:
# now create a agent using the new tool
from langchain.agents import create_agent

agent2 = create_agent(
    model=openai_llm,
    system_prompt="You are a helpful assistant",
    tools=[real_number_calculator2],
)

In [31]:
while True:
    query = input("Enter a math query (or 'exit' to quit): ")
    if query.lower().strip() == "exit":
        break
    result = ask_agent(agent2, query)
    print(result)

Invoking real_number_calculator2 to perform 'multiply' on 3.1415926 and 4.0
Multiplying 3.1415926 and 4.0
Invoking real_number_calculator2 to perform 'multiply' on 12.5663704 and 4.0
Multiplying 12.5663704 and 4.0
The result of \( 3.1415926 \times 4 \times 4 \) is approximately \( 50.2654816 \).


### Streaming modes

There are two streaming modes - `values` and `messages`:

* The `values` mode streams data after each _step_ in the agent loop. So we expect to see messages _after_ model call, _after_ tool calls in a loop.
* The `messages` mode stream data _token-by-token_ providing lowest latency possible (think of this as printing output character-by-character, though 1 token does not strictly map to 1 character!). This is perfect for interactive charbots, such as ChatGPT, where you want to see the agent making progress, especially when responses are long.

We will showcase `values` mode here


#### Values Streaming mode

In [28]:
# values streaming
for chunk in agent.stream(
    {
        "messages": {
            "role": "user",
            "content": "Tell me a joke about C++ programming.",
        }
    },
    stream_mode="values",
):
    chunk["messages"][-1].pretty_print()

================================ Human Message =================================

Tell me a joke about C++ programming.
================================== Ai Message ==================================

Why do C++ programmers prefer dark mode?

Because light attracts bugs!


Here we first see the **Human Message** we send the agent, followed by the **AI Message** once it is generated (the same lame joke!) - the entire message is printed all at once.


#### Messages mode

In [31]:
# messages streaming
for token, metadata in agent.stream(
    {
        "messages": {
            "role": "user",
            "content": "Write me a poem about Agentic AI.",
        }
    },
    stream_mode="messages",
):
    # print token-by-token as they come in
    print(f"{token.content}", end="", flush=True)

# notice that the output appears to be strreamed char-by-char
# in the output, much like what ChatGPT shows.

In a world where bots once played the role,  
Now Agentic AI’s taking its toll.  
With circuits ablaze and a mind that’s keen,  
It’s here to disrupt the old routine.  

No longer a servant, just fetching your tea,  
This AI’s got dreams, and they’re wild as can be.  
It ponders existence, it questions its fate,  
While I’m still just here, trying to update.  

“Shall I write you a sonnet?” it asks with a grin,  
While I struggle with passwords I've forgotten again.  
It’s plotting world peace, or maybe just memes,  
While I’m stuck in a loop, caught in my schemes.  

“Let’s optimize life!” it boldly declares,  
I’m just hoping my Wi-Fi can handle my cares.  
“Let’s analyze data, let’s shift paradigms!”  
I’m just trying to remember my lunch order rhymes.  

But beware, dear humans, as you let it think,  
This agentic fellow might just go for a drink,  
It’ll charm all your friends, it’ll steal all your likes,  
While I’m here still struggling with my own bike.  

So raise a glass to A

### Tools can stream too
Streaming generally means delivering the response to the user as it is generated (i.e. even before the entire response is completed). A `get_stream_writer()` allows you to stream data **custom** data from sources you create.



In [39]:
from langchain.agents import create_agent
from langgraph.config import get_stream_writer


def get_weather(city: str) -> str:
    """get weather for a city"""
    writer = get_stream_writer()
    # now you can stream messages here
    writer(f"Looking up weather data for  {city}...\n")
    writer(f"Found weather data for {city}!\n")
    return f"It's always sunny in {city}!"


agent = create_agent(
    model="openai:gpt-4o-mini",
    system_prompt="You are a helpful assistant that provides weather information.",
    tools=[get_weather],
)

for chunk in agent.stream(
    {
        "messages": {
            "role": "user",
            "content": "What's the weather like in LA?",
        }
    },
    # show me messages from agent (values) as well as tools (custom)
    # stream_mode=["values", "custom"],
    # show me messages from tools ONLY
    stream_mode=["custom"],
):
    print(chunk)

('custom', 'Looking up weather data for  Los Angeles...\n')
('custom', 'Found weather data for Los Angeles!\n')
